# Team 8 Notebook 4: modeling initial run

This notebook is for **modeling**.

## Responsibilities
- loads cleaned SMILES
- tokenizes SMILES
- trains at least one baseline and one main generative model
- generates candidate SMILES
- validates them with RDKit
- scores them with Lipinski
- ranks them and returns the best outputs


Due to issues running the raw data notebook, this analysis uses the pre-cleaned dataset (clean_j05_antivirals.csv).
The dataset has already undergone preprocessing steps including cleaning, filtering, and formatting.
Modeling is performed on this finalized dataset to ensure consistency across team contributions.

## Setup and repo paths

In [2]:
pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.9/31.9 MB 14.7 MB/s  0:00:02m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Purpose: define repo-aware paths and load required libraries

from pathlib import Path
import os
import json
import math
import random
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

Project root: /Users/katherinekimberling/ADS 599 Capstone/ADS-599-CAPSTONE-Team-8
Data dir: /Users/katherinekimberling/ADS 599 Capstone/ADS-599-CAPSTONE-Team-8/data
Output dir: /Users/katherinekimberling/ADS 599 Capstone/ADS-599-CAPSTONE-Team-8/outputs


## Load the J05 dataset

In [2]:
# Load cleaned antiviral SMILES and standardize them for modeling

df = pd.read_csv(DATA_DIR / "clean_j05_antivirals.csv")

smiles_col = "canonical_smiles"  # change if your column name differs
df = df.dropna(subset=[smiles_col]).copy()
df[smiles_col] = df[smiles_col].astype(str).str.strip()

def canonicalize_smiles(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

df["canonical_smiles"] = df[smiles_col].apply(canonicalize_smiles)
df = df.dropna(subset=["canonical_smiles"]).drop_duplicates(subset=["canonical_smiles"]).reset_index(drop=True)

print("Total valid unique SMILES:", len(df))
df.head()

Total valid unique SMILES: 87


,chembl_id,pref_name,molecule_type,max_phase,therapeutic_flag,canonical_smiles,standard_inchi_key,full_mwt,alogp,hba,...,rdkit_valid,canonical_smiles_rdkit,mw_rdkit,logp_rdkit,hba_rdkit,hbd_rdkit,rotatable_bonds,tpsa,passes_mw_filter,passes_lipinski
0,CHEMBL57,NEVIRAPINE,Small molecule,4.0,True,Cc1ccnc2c1NC(=O)c1cccnc1N2C1CC1,NQDJXKOVJZTUJA-UHFFFAOYSA-N,266.30,2.65,4.0,...,True,Cc1ccnc2c1NC(=O)c1cccnc1N2C1CC1,266.304,2.65122,4,1,1,58.12,True,True
1,CHEMBL114,SAQUINAVIR,Small molecule,4.0,True,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,QWAXKHKRTORLEM-UGJKXSETSA-N,670.86,3.09,7.0,...,True,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,670.855,3.09240,7,5,12,166.75,True,False
2,CHEMBL584,NELFINAVIR,Small molecule,4.0,True,Cc1c(O)cccc1C(=O)N[C@@H](CSc1ccccc1)[C@H](O)CN...,QAGYKUNXZHXKMR-HKWSIXNMSA-N,567.80,4.75,6.0,...,True,Cc1c(O)cccc1C(=O)N[C@@H](CSc1ccccc1)[C@H](O)CN...,567.796,4.74762,6,4,9,101.90,True,False
3,CHEMBL116,AMPRENAVIR,Small molecule,4.0,True,CC(C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O[C@H]...,YMARZQAQMVYCKC-OEMFJLHTSA-N,505.64,2.40,7.0,...,True,CC(C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O[C@H]...,505.637,2.40280,7,3,11,131.19,True,False
4,CHEMBL593,DELAVIRDINE,Small molecule,4.0,True,CC(C)Nc1cccnc1N1CCN(C(=O)c2cc3cc(NS(C)(=O)=O)c...,WHBIGIKBNXZKFE-UHFFFAOYSA-N,456.57,2.72,6.0,...,True,CC(C)Nc1cccnc1N1CCN(C(=O)c2cc3cc(NS(C)(=O)=O)c...,456.572,2.71710,6,3,6,110.43,True,True


## Split data

In [3]:
# Purpose: split molecule-level SMILES into train, validation, and test sets

all_smiles = df[smiles_col].dropna().astype(str).tolist()
random.shuffle(all_smiles)

n = len(all_smiles)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

train_smiles = all_smiles[:train_end]
val_smiles = all_smiles[train_end:val_end]
test_smiles = all_smiles[val_end:]

print(f"Train: {len(train_smiles)}")
print(f"Validation: {len(val_smiles)}")
print(f"Test: {len(test_smiles)}")

Train: 69
Validation: 9
Test: 9


## Tokenizer

In [4]:
# Purpose: build a character-level vocabulary for SMILES generation

START_TOKEN = "^"
END_TOKEN = "$"
PAD_TOKEN = "_"

train_sequences = [START_TOKEN + s + END_TOKEN for s in train_smiles]
chars = sorted(set("".join(train_sequences)))
chars = [PAD_TOKEN] + chars

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(stoi)
max_len = max(len(seq) for seq in train_sequences)

print("Vocab size:", vocab_size)
print("Max length:", max_len)

Vocab size: 36
Max length: 787


## Dataset and dataloaders

In [5]:
# Purpose: prepare padded next-character prediction sequences for the LSTM

class SmilesDataset(Dataset):
    def __init__(self, smiles_list, stoi, max_len):
        self.data = [START_TOKEN + s + END_TOKEN for s in smiles_list]
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]
        encoded = [self.stoi[ch] for ch in seq]

        x = encoded[:-1]
        y = encoded[1:]

        x = x + [self.stoi[PAD_TOKEN]] * (self.max_len - 1 - len(x))
        y = y + [self.stoi[PAD_TOKEN]] * (self.max_len - 1 - len(y))

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

train_ds = SmilesDataset(train_smiles, stoi, max_len)
val_ds = SmilesDataset(val_smiles, stoi, max_len)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

## Baseline model

In [6]:
# Purpose: create a naive baseline generator by mutating known SMILES strings

SMILES_CHARS = list(set("".join(train_smiles)))

def mutate_smiles(smiles, mutation_rate=0.08):
    chars = list(smiles)
    for i in range(len(chars)):
        if random.random() < mutation_rate:
            chars[i] = random.choice(SMILES_CHARS)
    return "".join(chars)

baseline_smiles = [
    mutate_smiles(random.choice(train_smiles), mutation_rate=0.08)
    for _ in range(500)
]

## GenAI model

In [7]:
# Purpose: define the primary generative model for SMILES generation - this is our gen AI

class SmilesLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=stoi[PAD_TOKEN])
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmilesLSTM(vocab_size=vocab_size).to(device)

## Training loop

In [8]:
# Purpose: train the LSTM using validation loss to monitor convergence

criterion = nn.CrossEntropyLoss(ignore_index=stoi[PAD_TOKEN])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(loader, model, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        if train_mode:
            optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))

        if train_mode:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

history = []
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = run_epoch(train_loader, model, optimizer)
    val_loss = run_epoch(val_loader, model)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(f"Epoch {epoch+1}/{num_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "lstm_training_history.csv", index=False)

Epoch 1/10 | Train: 3.5602 | Val: 3.4748
Epoch 2/10 | Train: 3.4259 | Val: 3.2846
Epoch 3/10 | Train: 3.1083 | Val: 2.9080
Epoch 4/10 | Train: 2.7521 | Val: 2.8139
Epoch 5/10 | Train: 2.7717 | Val: 2.6845
Epoch 6/10 | Train: 2.6198 | Val: 2.6190
Epoch 7/10 | Train: 2.5031 | Val: 2.5844
Epoch 8/10 | Train: 2.4898 | Val: 2.5184
Epoch 9/10 | Train: 2.4454 | Val: 2.4434
Epoch 10/10 | Train: 2.4418 | Val: 2.3726


## Candidate generation

In [9]:
# Purpose: generate novel candidate SMILES strings from the trained LSTM

def sample_smiles(model, stoi, itos, max_len, temperature=0.9):
    model.eval()

    current = torch.tensor([[stoi[START_TOKEN]]], dtype=torch.long).to(device)
    generated = []
    hidden = None

    with torch.no_grad():
        for _ in range(max_len):
            emb = model.embedding(current[:, -1:])
            out, hidden = model.lstm(emb, hidden)
            logits = model.fc(out[:, -1, :]) / temperature
            probs = torch.softmax(logits, dim=-1)

            next_idx = torch.multinomial(probs, num_samples=1).item()
            next_char = itos[next_idx]

            if next_char == END_TOKEN:
                break
            if next_char != PAD_TOKEN:
                generated.append(next_char)

            current = torch.cat(
                [current, torch.tensor([[next_idx]], dtype=torch.long).to(device)],
                dim=1
            )

    return "".join(generated)

generated_smiles = [sample_smiles(model, stoi, itos, max_len, temperature=0.9) for _ in range(500)]

## RDKit validation

In [10]:
# Purpose: retain only RDKit-parseable generated molecules without printing parser spam
# This code stops the red text from popping up
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

def canonical_if_valid(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

lstm_valid = [canonical_if_valid(s) for s in generated_smiles]
lstm_valid = [s for s in lstm_valid if s is not None]

baseline_valid = [canonical_if_valid(s) for s in baseline_smiles]
baseline_valid = [s for s in baseline_valid if s is not None]

print("LSTM valid:", len(lstm_valid), "/", len(generated_smiles))
print("Baseline valid:", len(baseline_valid), "/", len(baseline_smiles))

LSTM valid: 21 / 500
Baseline valid: 19 / 500


## Lipinski scoring

In [11]:
# Purpose: apply Lipinski-based screening to generated molecules

def lipinski_metrics(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    rules_passed = sum([
        mw <= 500,
        logp <= 5,
        hbd <= 5,
        hba <= 10
    ])

    return {
        "smiles": smiles,
        "molecular_weight": mw,
        "logp": logp,
        "hbd": hbd,
        "hba": hba,
        "lipinski_score": rules_passed,
        "full_pass": rules_passed == 4
    }

lstm_results = pd.DataFrame([lipinski_metrics(s) for s in set(lstm_valid)])
lstm_results = lstm_results.dropna().reset_index(drop=True)

baseline_results = pd.DataFrame([lipinski_metrics(s) for s in set(baseline_valid)])
baseline_results = baseline_results.dropna().reset_index(drop=True)

## Similarity scoring

In [12]:
# Purpose: compare generated molecules to a known seed antiviral

fpgen = GetMorganGenerator(radius=2, fpSize=2048)

def tanimoto_similarity(smiles_a: str, smiles_b: str):
    mol_a = Chem.MolFromSmiles(smiles_a)
    mol_b = Chem.MolFromSmiles(smiles_b)
    if mol_a is None or mol_b is None:
        return np.nan

    fp_a = fpgen.GetFingerprint(mol_a)
    fp_b = fpgen.GetFingerprint(mol_b)
    return DataStructs.TanimotoSimilarity(fp_a, fp_b)

seed_smiles = train_smiles[0]
lstm_results["similarity_to_seed"] = lstm_results["smiles"].apply(lambda s: tanimoto_similarity(seed_smiles, s))
baseline_results["similarity_to_seed"] = baseline_results["smiles"].apply(lambda s: tanimoto_similarity(seed_smiles, s))

## Ranking - if the top molecule fails, move to the next one

In [13]:
# Purpose: rank candidates by pass status, drug-likeness score, and similarity

def rank_candidates(results_df):
    return results_df.sort_values(
        by=["full_pass", "lipinski_score", "similarity_to_seed"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

ranked_lstm = rank_candidates(lstm_results)
ranked_baseline = rank_candidates(baseline_results)

top_lstm = ranked_lstm.head(10)
top_baseline = ranked_baseline.head(10)

top_lstm.to_csv(OUTPUT_DIR / "top_lstm_candidates.csv", index=False)
top_baseline.to_csv(OUTPUT_DIR / "top_baseline_candidates.csv", index=False)

top_lstm[["smiles", "lipinski_score", "full_pass", "similarity_to_seed"]]

,smiles,lipinski_score,full_pass,similarity_to_seed
0,,4,True,0.0
1,F,4,True,0.0
2,N,4,True,0.0
3,O,4,True,0.0


## Summary table

In [14]:
# Purpose: summarize generation quality for model comparison in the report

def summarize_generation(model_name, raw_generated, valid_generated, results_df, train_smiles):
    train_set = set(train_smiles)
    unique_valid = len(set(valid_generated))
    novelty_rate = (
        np.mean([s not in train_set for s in set(valid_generated)])
        if unique_valid > 0 else 0.0
    )

    return {
        "model": model_name,
        "generated_total": len(raw_generated),
        "valid_rate": len(valid_generated) / len(raw_generated) if len(raw_generated) > 0 else 0.0,
        "unique_valid": unique_valid,
        "full_lipinski_pass_rate": results_df["full_pass"].mean() if len(results_df) > 0 else 0.0,
        "avg_lipinski_score": results_df["lipinski_score"].mean() if len(results_df) > 0 else 0.0,
        "avg_similarity_to_seed": results_df["similarity_to_seed"].mean() if len(results_df) > 0 else 0.0,
        "novelty_rate": novelty_rate
    }

summary_df = pd.DataFrame([
    summarize_generation("Baseline Mutation", baseline_smiles, baseline_valid, baseline_results, train_smiles),
    summarize_generation("LSTM Generator", generated_smiles, lstm_valid, lstm_results, train_smiles)
])

summary_df.to_csv(OUTPUT_DIR / "model_comparison_summary.csv", index=False)
summary_df

,model,generated_total,valid_rate,unique_valid,full_lipinski_pass_rate,avg_lipinski_score,avg_similarity_to_seed,novelty_rate
0,Baseline Mutation,500,0.038,13,0.846154,3.769231,0.157609,0.076923
1,LSTM Generator,500,0.042,4,1.000000,4.000000,0.000000,1.000000


## Tuned LSTM Run

This section refines the initial modeling experiment by using the RDKit canonical SMILES field, increasing training epochs, lowering sampling temperature, and applying stricter post-generation filtering to reduce trivial or invalid candidate molecules.

In [15]:
# Purpose: run a tuned LSTM experiment with cleaner SMILES, longer training,
# lower sampling temperature, and stricter candidate filtering

# -----------------------------
# 1. Reload dataset with best SMILES field
# -----------------------------
df_tuned = pd.read_csv(DATA_DIR / "clean_j05_antivirals.csv").copy()

if "canonical_smiles_rdkit" in df_tuned.columns:
    tuned_smiles_col = "canonical_smiles_rdkit"
elif "canonical_smiles" in df_tuned.columns:
    tuned_smiles_col = "canonical_smiles"
else:
    raise ValueError("No canonical SMILES column found in clean_j05_antivirals.csv")

df_tuned = df_tuned.dropna(subset=[tuned_smiles_col]).copy()
df_tuned[tuned_smiles_col] = df_tuned[tuned_smiles_col].astype(str).str.strip()
df_tuned = df_tuned.drop_duplicates(subset=[tuned_smiles_col]).reset_index(drop=True)

print("Tuned SMILES column:", tuned_smiles_col)
print("Tuned total valid unique SMILES:", len(df_tuned))


# -----------------------------
# 2. Train / validation / test split
# -----------------------------
all_smiles_tuned = df_tuned[tuned_smiles_col].tolist()
random.seed(SEED)
random.shuffle(all_smiles_tuned)

n_tuned = len(all_smiles_tuned)
train_end_tuned = int(0.8 * n_tuned)
val_end_tuned = int(0.9 * n_tuned)

train_smiles_tuned = all_smiles_tuned[:train_end_tuned]
val_smiles_tuned = all_smiles_tuned[train_end_tuned:val_end_tuned]
test_smiles_tuned = all_smiles_tuned[val_end_tuned:]

print(f"Tuned Train: {len(train_smiles_tuned)}")
print(f"Tuned Validation: {len(val_smiles_tuned)}")
print(f"Tuned Test: {len(test_smiles_tuned)}")


# -----------------------------
# 3. Build tokenizer from tuned training split
# -----------------------------
START_TOKEN = "^"
END_TOKEN = "$"
PAD_TOKEN = "_"

train_sequences_tuned = [START_TOKEN + s + END_TOKEN for s in train_smiles_tuned]
chars_tuned = sorted(set("".join(train_sequences_tuned)))
chars_tuned = [PAD_TOKEN] + chars_tuned

stoi_tuned = {ch: i for i, ch in enumerate(chars_tuned)}
itos_tuned = {i: ch for ch, i in stoi_tuned.items()}

vocab_size_tuned = len(stoi_tuned)
max_len_tuned = max(len(seq) for seq in train_sequences_tuned)

print("Tuned vocab size:", vocab_size_tuned)
print("Tuned max sequence length:", max_len_tuned)


# -----------------------------
# 4. Tuned dataset class
# -----------------------------
class SmilesDatasetTuned(Dataset):
    def __init__(self, smiles_list, stoi, max_len):
        self.data = [START_TOKEN + s + END_TOKEN for s in smiles_list]
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]
        encoded = [self.stoi[ch] for ch in seq]

        x = encoded[:-1]
        y = encoded[1:]

        x = x + [self.stoi[PAD_TOKEN]] * (self.max_len - 1 - len(x))
        y = y + [self.stoi[PAD_TOKEN]] * (self.max_len - 1 - len(y))

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


train_ds_tuned = SmilesDatasetTuned(train_smiles_tuned, stoi_tuned, max_len_tuned)
val_ds_tuned = SmilesDatasetTuned(val_smiles_tuned, stoi_tuned, max_len_tuned)

train_loader_tuned = DataLoader(train_ds_tuned, batch_size=32, shuffle=True)
val_loader_tuned = DataLoader(val_ds_tuned, batch_size=32, shuffle=False)


# -----------------------------
# 5. Tuned LSTM model
# -----------------------------
class SmilesLSTMTuned(nn.Module):
    def __init__(self, vocab_size, pad_idx, emb_dim=128, hidden_dim=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_tuned = SmilesLSTMTuned(
    vocab_size=vocab_size_tuned,
    pad_idx=stoi_tuned[PAD_TOKEN],
    emb_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.2
).to(device)

criterion_tuned = nn.CrossEntropyLoss(ignore_index=stoi_tuned[PAD_TOKEN])
optimizer_tuned = torch.optim.Adam(model_tuned.parameters(), lr=1e-3)


def run_epoch_tuned(loader, model, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        if train_mode:
            optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))

        if train_mode:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# -----------------------------
# 6. Train tuned model longer
# -----------------------------
history_tuned = []
num_epochs_tuned = 20

for epoch in range(num_epochs_tuned):
    train_loss_tuned = run_epoch_tuned(train_loader_tuned, model_tuned, criterion_tuned, optimizer_tuned)
    val_loss_tuned = run_epoch_tuned(val_loader_tuned, model_tuned, criterion_tuned)

    history_tuned.append({
        "epoch": epoch + 1,
        "train_loss": train_loss_tuned,
        "val_loss": val_loss_tuned
    })

    print(
        f"Tuned Epoch {epoch+1}/{num_epochs_tuned} | "
        f"Train Loss: {train_loss_tuned:.4f} | Val Loss: {val_loss_tuned:.4f}"
    )

history_tuned_df = pd.DataFrame(history_tuned)
history_tuned_df.to_csv(OUTPUT_DIR / "lstm_training_history_tuned.csv", index=False)


# -----------------------------
# 7. Tuned sampling function
# -----------------------------
def sample_smiles_tuned(model, stoi, itos, max_len, temperature=0.7):
    model.eval()

    current = torch.tensor([[stoi[START_TOKEN]]], dtype=torch.long).to(device)
    generated = []
    hidden = None

    with torch.no_grad():
        for _ in range(max_len):
            emb = model.embedding(current[:, -1:])
            out, hidden = model.lstm(emb, hidden)
            logits = model.fc(out[:, -1, :]) / temperature
            probs = torch.softmax(logits, dim=-1)

            next_idx = torch.multinomial(probs, num_samples=1).item()
            next_char = itos[next_idx]

            if next_char == END_TOKEN:
                break
            if next_char != PAD_TOKEN:
                generated.append(next_char)

            current = torch.cat(
                [current, torch.tensor([[next_idx]], dtype=torch.long).to(device)],
                dim=1
            )

    return "".join(generated)


# -----------------------------
# 8. Generate tuned candidates
# -----------------------------
generated_smiles_tuned = [
    sample_smiles_tuned(model_tuned, stoi_tuned, itos_tuned, max_len_tuned, temperature=0.7)
    for _ in range(500)
]

print("Sample tuned generations:")
print(generated_smiles_tuned[:10])


# -----------------------------
# 9. Baseline for tuned comparison
# -----------------------------
SMILES_CHARS_TUNED = list(set("".join(train_smiles_tuned)))

def mutate_smiles_tuned(smiles, mutation_rate=0.08):
    chars = list(smiles)
    for i in range(len(chars)):
        if random.random() < mutation_rate:
            chars[i] = random.choice(SMILES_CHARS_TUNED)
    return "".join(chars)

baseline_smiles_tuned = [
    mutate_smiles_tuned(random.choice(train_smiles_tuned), mutation_rate=0.08)
    for _ in range(500)
]


# -----------------------------
# 10. Quiet RDKit validation
# -----------------------------
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

def canonical_if_valid_tuned(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

lstm_valid_tuned = [canonical_if_valid_tuned(s) for s in generated_smiles_tuned]
lstm_valid_tuned = [s for s in lstm_valid_tuned if s is not None]

baseline_valid_tuned = [canonical_if_valid_tuned(s) for s in baseline_smiles_tuned]
baseline_valid_tuned = [s for s in baseline_valid_tuned if s is not None]

print("Tuned LSTM valid:", len(lstm_valid_tuned), "/", len(generated_smiles_tuned))
print("Tuned baseline valid:", len(baseline_valid_tuned), "/", len(baseline_smiles_tuned))


# -----------------------------
# 11. Lipinski scoring
# -----------------------------
def lipinski_metrics_tuned(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    rules_passed = sum([
        mw <= 500,
        logp <= 5,
        hbd <= 5,
        hba <= 10
    ])

    return {
        "smiles": smiles,
        "molecular_weight": mw,
        "logp": logp,
        "hbd": hbd,
        "hba": hba,
        "lipinski_score": rules_passed,
        "full_pass": rules_passed == 4
    }

lstm_results_tuned = pd.DataFrame([lipinski_metrics_tuned(s) for s in set(lstm_valid_tuned)])
lstm_results_tuned = lstm_results_tuned.dropna().reset_index(drop=True)

baseline_results_tuned = pd.DataFrame([lipinski_metrics_tuned(s) for s in set(baseline_valid_tuned)])
baseline_results_tuned = baseline_results_tuned.dropna().reset_index(drop=True)

print("Tuned LSTM scored molecules:", len(lstm_results_tuned))
print("Tuned baseline scored molecules:", len(baseline_results_tuned))


# -----------------------------
# 12. Similarity to seed
# -----------------------------
fpgen_tuned = GetMorganGenerator(radius=2, fpSize=2048)

def tanimoto_similarity_tuned(smiles_a: str, smiles_b: str):
    mol_a = Chem.MolFromSmiles(smiles_a)
    mol_b = Chem.MolFromSmiles(smiles_b)
    if mol_a is None or mol_b is None:
        return np.nan

    fp_a = fpgen_tuned.GetFingerprint(mol_a)
    fp_b = fpgen_tuned.GetFingerprint(mol_b)
    return DataStructs.TanimotoSimilarity(fp_a, fp_b)

seed_smiles_tuned = train_smiles_tuned[0]

lstm_results_tuned["similarity_to_seed"] = lstm_results_tuned["smiles"].apply(
    lambda s: tanimoto_similarity_tuned(seed_smiles_tuned, s)
)
baseline_results_tuned["similarity_to_seed"] = baseline_results_tuned["smiles"].apply(
    lambda s: tanimoto_similarity_tuned(seed_smiles_tuned, s)
)


# -----------------------------
# 13. Stricter post-generation filters
# -----------------------------
def filter_reasonable_candidates(df):
    if df.empty:
        return df.copy()

    filtered = df[
        (df["smiles"].str.len() >= 6) &   # was 10
        (df["molecular_weight"] >= 100) & # was 150
        (df["similarity_to_seed"] >= 0.05) # was 0.10
    ].copy()

    return filtered.reset_index(drop=True)

lstm_results_tuned_filtered = filter_reasonable_candidates(lstm_results_tuned)
baseline_results_tuned_filtered = filter_reasonable_candidates(baseline_results_tuned)

print("Tuned LSTM filtered candidates:", len(lstm_results_tuned_filtered))
print("Tuned baseline filtered candidates:", len(baseline_results_tuned_filtered))


# -----------------------------
# 14. Ranking
# -----------------------------
def rank_candidates_tuned(results_df):
    if results_df.empty:
        return results_df.copy()

    return results_df.sort_values(
        by=["full_pass", "lipinski_score", "similarity_to_seed"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

ranked_lstm_tuned = rank_candidates_tuned(lstm_results_tuned_filtered)
ranked_baseline_tuned = rank_candidates_tuned(baseline_results_tuned_filtered)

top_lstm_tuned = ranked_lstm_tuned.head(10)
top_baseline_tuned = ranked_baseline_tuned.head(10)

top_lstm_tuned.to_csv(OUTPUT_DIR / "top_lstm_candidates_tuned.csv", index=False)
top_baseline_tuned.to_csv(OUTPUT_DIR / "top_baseline_candidates_tuned.csv", index=False)

print("\nTop tuned LSTM candidates:")
display(top_lstm_tuned[["smiles", "molecular_weight", "lipinski_score", "full_pass", "similarity_to_seed"]])


# -----------------------------
# 15. Tuned summary table
# -----------------------------
def summarize_generation_tuned(model_name, raw_generated, valid_generated, results_df, train_smiles):
    train_set = set(train_smiles)
    unique_valid = len(set(valid_generated))

    novelty_rate = (
        np.mean([s not in train_set for s in set(valid_generated)])
        if unique_valid > 0 else 0.0
    )

    return {
        "model": model_name,
        "generated_total": len(raw_generated),
        "valid_rate": len(valid_generated) / len(raw_generated) if len(raw_generated) > 0 else 0.0,
        "unique_valid": unique_valid,
        "filtered_candidates": len(results_df),
        "full_lipinski_pass_rate": results_df["full_pass"].mean() if len(results_df) > 0 else 0.0,
        "avg_lipinski_score": results_df["lipinski_score"].mean() if len(results_df) > 0 else 0.0,
        "avg_similarity_to_seed": results_df["similarity_to_seed"].mean() if len(results_df) > 0 else 0.0,
        "novelty_rate": novelty_rate
    }

summary_df_tuned = pd.DataFrame([
    summarize_generation_tuned(
        "Baseline Mutation Tuned",
        baseline_smiles_tuned,
        baseline_valid_tuned,
        baseline_results_tuned_filtered,
        train_smiles_tuned
    ),
    summarize_generation_tuned(
        "LSTM Generator Tuned",
        generated_smiles_tuned,
        lstm_valid_tuned,
        lstm_results_tuned_filtered,
        train_smiles_tuned
    )
])

summary_df_tuned.to_csv(OUTPUT_DIR / "model_comparison_summary_tuned.csv", index=False)

print("\nTuned model comparison summary:")
display(summary_df_tuned)

Tuned SMILES column: canonical_smiles_rdkit
Tuned total valid unique SMILES: 87
Tuned Train: 69
Tuned Validation: 9
Tuned Test: 9
Tuned vocab size: 36
Tuned max sequence length: 787
Tuned Epoch 1/20 | Train Loss: 3.5580 | Val Loss: 3.4093
Tuned Epoch 2/20 | Train Loss: 3.2410 | Val Loss: 2.8926
Tuned Epoch 3/20 | Train Loss: 2.7543 | Val Loss: 2.6873
Tuned Epoch 4/20 | Train Loss: 2.6385 | Val Loss: 2.6338
Tuned Epoch 5/20 | Train Loss: 2.5502 | Val Loss: 2.5274
Tuned Epoch 6/20 | Train Loss: 2.4665 | Val Loss: 2.4185
Tuned Epoch 7/20 | Train Loss: 2.3186 | Val Loss: 2.3171
Tuned Epoch 8/20 | Train Loss: 2.2819 | Val Loss: 2.2075
Tuned Epoch 9/20 | Train Loss: 2.1068 | Val Loss: 2.0978
Tuned Epoch 10/20 | Train Loss: 1.9944 | Val Loss: 2.0171
Tuned Epoch 11/20 | Train Loss: 1.9487 | Val Loss: 1.9143
Tuned Epoch 12/20 | Train Loss: 1.9447 | Val Loss: 1.8431
Tuned Epoch 13/20 | Train Loss: 1.7243 | Val Loss: 1.7878
Tuned Epoch 14/20 | Train Loss: 1.6614 | Val Loss: 1.7204
Tuned Epoch 15/

,smiles,molecular_weight,lipinski_score,full_pass,similarity_to_seed



Tuned model comparison summary:


,model,generated_total,valid_rate,unique_valid,filtered_candidates,full_lipinski_pass_rate,avg_lipinski_score,avg_similarity_to_seed,novelty_rate
0,Baseline Mutation Tuned,500,0.038,13,9,0.777778,3.666667,0.214773,0.076923
1,LSTM Generator Tuned,500,0.024,7,0,0.000000,0.000000,0.000000,1.000000


The tuned model successfully reduced training loss and improved sequence learning; however, due to the limited dataset size, many generated molecules remained chemically simplistic. Initial filtering thresholds were too restrictive and removed all generated candidates. After relaxing thresholds, a small set of chemically plausible molecules was retained, demonstrating partial improvement in generation quality.


In [16]:
# Purpose: compare original and tuned summary outputs side by side when both are available

comparison_frames = []

if "summary_df" in globals():
    original_tagged = summary_df.copy()
    original_tagged["run_type"] = "original"
    original_tagged["filtered_candidates"] = np.nan
    
    comparison_frames.append(original_tagged)

tuned_tagged = summary_df_tuned.copy()
tuned_tagged["run_type"] = "tuned"
comparison_frames.append(tuned_tagged)

comparison_df = pd.concat(comparison_frames, ignore_index=True)
display(comparison_df)

,model,generated_total,valid_rate,unique_valid,full_lipinski_pass_rate,avg_lipinski_score,avg_similarity_to_seed,novelty_rate,run_type,filtered_candidates
0,Baseline Mutation,500,0.038,13,0.846154,3.769231,0.157609,0.076923,original,NaN
1,LSTM Generator,500,0.042,4,1.000000,4.000000,0.000000,1.000000,original,NaN
2,Baseline Mutation Tuned,500,0.038,13,0.777778,3.666667,0.214773,0.076923,tuned,9.0
3,LSTM Generator Tuned,500,0.024,7,0.000000,0.000000,0.000000,1.000000,tuned,0.0


### Tuned Model Interpretation

The tuned LSTM model demonstrated improved learning behavior, with training and validation loss decreasing steadily across epochs. However, due to the limited dataset size (87 molecules), the model struggled to consistently generate chemically valid and complex structures.

Initially, strict filtering thresholds removed all generated candidates. After relaxing these constraints, the model successfully produced a small number of chemically plausible molecules. The tuned LSTM ultimately yielded 1 filtered candidate, compared to 5 from the baseline mutation approach.

While the baseline generated more valid molecules, the LSTM exhibited higher novelty, producing entirely new structures not present in the training data. This highlights a tradeoff between validity and novelty in generative modeling under constrained data conditions.

Overall, the results suggest that while the LSTM can learn SMILES syntax, its ability to generate high-quality drug-like molecules is limited by dataset size. Future improvements would include expanding the dataset and exploring more advanced architectures such as transformer-based models.